# Harvesting under a strong Allee effect

<div style="border-left:5px solid #176b87;background:#176b8710;padding:12px"><strong>Read, compute, interpret</strong><br>Five linked work packages study constant catch, proportional effort, collapse boundaries, forward PINNs and inverse inference. All numerical examples are synthetic.</div>

1. WP01 derives scaling, equilibria and stability.
2. WP02 compares explicit harvesting protocols and follows continuous histories with declared restocking.
3. WP03 classifies basins and checks finite extinction by independent quadrature and solvers.
4. WP04 tests forward PINNs against numerical references.
5. WP05 fits identifiable parameter subsets using noisy observations, a classical baseline and held-out windows.

Equations and algorithms precede code; plots are followed by interpretation and limitations. The shared classical APIs use one threshold/branch provider. A small population in B is not finite extinction; A's zero continuation is a declared shutdown policy. Good prediction does not guarantee accurate parameter recovery.

The standalone export includes readable source/config/data/evidence. Restart the kernel and Run All. Recorded final experiments are distinguished from live demonstrations. The inverse reporting fixes from PR #10 commit `00393cd` are integrated. Section 5 explains the legacy training-status label and the separate L-BFGS outcome. The Overleaf report, presentation/rehearsal and human approvals are maintained separately.


In [ ]:
import os
from pathlib import Path

import mm_allee

ROOT = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "pyproject.toml").exists() and (path / "src/mm_allee").exists()
    ),
    None,
)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the project checkout")
os.chdir(ROOT)
print("Package:", mm_allee.__version__)

## 1. Foundations and Model A: steady populations under a fixed catch

**Question.** With the same catch every year, where can the population stay steady, which equilibrium points are stable, and what is the largest catch before collapse is certain?

**Model A.** A well-mixed population $N(t)$ with growth rate $r$, carrying capacity $K$ and Allee threshold $A$ ($0<A<K$), harvested at a constant rate $H$:
$$\frac{dN}{dt}=rN\Big(1-\frac{N}{K}\Big)\Big(\frac{N}{A}-1\Big)-H.$$
Below $A$ growth is negative, so a small population shrinks even without fishing. We assume deterministic dynamics and constant parameters, and stop a simulation when $N$ reaches 0, since an extinct stock cannot be fished.

**Scaling.** With $x=N/K$, $\tau=rt$, $a=A/K$ and $q=H/(rK)$,
$$\frac{dx}{d\tau}=f(x)=G(x)-q,\qquad G(x)=x(1-x)\Big(\frac{x}{a}-1\Big),$$
so only $a$ and $q$ matter. We use $a=0.2$, for example $r=0.5$/year, $K=1000$ t and $A=200$ t as in Section 2.

### 1.1 Steady states and stability

A steady state $x_e$ solves $f(x_e)=0$. It is **stable** if $f>0$ just below it and $f<0$ just above it, so a small disturbance dies out. With the opposite signs it is **unstable**, and with the same sign on both sides it is **semistable**.

**Logistic baseline** ($f=x(1-x)-q$, no Allee effect). For $q<\tfrac14$ the steady states $x_\pm=\tfrac12\big(1\pm\sqrt{1-4q}\big)$ are unstable (lower) and stable (upper). They merge at $q=\tfrac14$, so the largest catch is $H_c=rK/4$. Without harvest, zero is unstable and any small population recovers.

In [ ]:
import matplotlib.pyplot as plt

from mm_allee.equilibria import equilibria_logistic

for q in [0.0, 0.1, 0.25, 0.3]:
    states = [f"x={e.x:.3f} ({e.stability})" for e in equilibria_logistic(q)]
    print(f"q={q}:", ", ".join(states) or "none, the population collapses")

### 1.2 Model A: the growth curve and the maximum catch

$G$ is negative on $(0,a)$ and forms a hill on $(a,1)$. Setting $G'(x)=0$ gives the top of the hill,
$$x^*=\frac{1+a+\sqrt{1-a+a^2}}{3},\qquad q_c=G(x^*).$$
Steady states are where the line at height $q$ meets $G$:

- **$q=0$:** $0$ (stable), $a$ (unstable, the minimum viable population) and $1$ (stable).
- **$0<q<q_c$:** $x_-$ between $a$ and $x^*$ (unstable) and $x_+$ between $x^*$ and 1 (stable). Below $x_-$ the population dies out; above it, it settles at $x_+$.
- **$q=q_c$:** one semistable point at $x^*$, the fold.
- **$q>q_c$:** none, so every population collapses.

For $q>0$, $f(0)=-q<0$, so zero is  no longer an equilibrium point. If the initial population is below the unstable equilibrium (x_-), it decreases and reaches zero in finite time. For $a=0.2$, $x^*\approx0.706$ and $q_c\approx0.525$, a maximum catch of about 263 t/year in the example.

### 1.3 Finding the steady states numerically

**Algorithm.**
1. Compute $x^*$ and $q_c$.
2. If $q>q_c$ there is no positive steady state; if $q=q_c$ there is one, at $x^*$.
3. If $0<q<q_c$, $f$ changes sign once on $[a,x^*]$ and once on $[x^*,1]$. Find each root with Brent's method, a standard root finder that narrows an interval around a sign change.
4. Label each root from the signs of $f$ on either side, as in 1.1.

In [ ]:
from mm_allee.equilibria import equilibria_a, fold_a

a = 0.2
x_star, q_c = fold_a(a)
print(f"x* = {x_star:.4f}, q_c = {q_c:.4f}\n")
for q in [0.0, 0.3, q_c, 0.6]:
    states = [f"x={e.x:.4f} ({e.stability})" for e in equilibria_a(a, q)]
    print(f"q={q:.4f}:", ", ".join(states) or "none, the population collapses")

### 1.4 Growth curves and the branch diagram

Filled circles are stable, open circles unstable and half-filled semistable. Arrows on the phase lines show whether the population grows or shrinks.

In [ ]:
from mm_allee.equilibria_plots import branch_figure, perturbation_figure, production_figure

fig = production_figure(a=0.2, logistic_q=(0.1, 0.25, 0.35), allee_q=(0.25 * q_c, q_c, 1.2 * q_c))
plt.show()

**Interpretation.** As the constant harvesting rate increases, the stable equilibrium (solid curve) and the unstable equilibrium (dashed curve), which acts as the collapse threshold, move closer together. They eventually meet at the critical point $(q_c, x^*)$. Beyond $q_c$, no positive equilibrium exists, and the population eventually becomes extinct.

In [ ]:
fig = branch_figure(a=0.2)
plt.show()

**Interpretation.** As the catch rises, the stable branch (solid) and the collapse threshold (dashed) move together and meet at the fold $(q_c,x^*)$, the gap between the equilibrium points decreases as the harvesting rate approaches its critical value.

### 1.5 Checking stability by simulation

At $q=0.3$, start $\delta=0.01$ above and below each steady state and solve the ODE (RK45, stopping at zero).

In [ ]:
fig = perturbation_figure(a=0.2, q=0.3, delta=0.01, tau_end=12.0)
plt.show()

Both starts near $x_+$ return to it. Just above $x_-$ the population recovers, and just below it reaches zero at $\tau\approx2.9$, as the stability labels predict.

<div style="border-left:5px solid #1baf7a;background:#1baf7a14;padding:10px 14px;border-radius:4px"><strong>Main result</strong></div>

As the constant harvesting rate increases, the stable and unstable equilibrium points move closer together and eventually merge at the critical harvesting rate (q_c). A population starting near the stable equilibrium can still collapse if a sufficiently large decrease takes it below the unstable equilibrium. Therefore, harvesting close to (q_c) leaves the population more vulnerable to disturbances.

**Limitations.** No randomness (which matters most near the threshold), a simple cubic growth law, constant parameters, and an exact catch whenever $N>0$. The numbers are illustrative, not from a real stock.

## 2. Does a smaller catch prevent collapse?

**Model A takes a fixed amount each year. Model B removes an amount proportional to the current population.**
For example, At a population of 100 tonnes, a harvesting rate of 10% per year corresponds to a catch rate of 10 tonnes/year,
but only 2 tonnes from a 20-tonne population.

We answer three questions:
1. Which population levels can remain steady under Model B?
2. How can we compare the two harvesting rules fairly?
3. Does reducing harvesting after collapse allow recovery?

These are **synthetic examples**, not observations of a real fish stock.

### 2.1 Steady populations under Model B

Use the same scaled variables as Section 1: $x=N/K$ is population relative to carrying
capacity, $a=A/K$ is the Allee threshold, $\tau=rt$ is scaled time, and $e=h/r$ is harvesting effort.

$$
\frac{dx}{d\tau}=x\left[(1-x)(x/a-1)-e\right]
$$

A steady population makes the right-hand side zero. One solution is $x=0$.
For the other solutions, divide by $x$ and rearrange:

$$
x^2-(1+a)x+a(1+e)=0
$$

The two roots are

$$
x_{\pm}=\frac{1+a\pm\sqrt{(1-a)^2-4ae}}{2}
$$

The lower equilibrium \(x_-\) represents the minimum population required for recovery under the given harvesting effort. When harvesting increases, this threshold rises above the original Allee threshold \(a\), making recovery more difficult.
They meet at the critical effort

$$
e_c=\frac{(1-a)^2}{4a}
$$

- **Below $e_c$:** the lower root $x_-$ is the recovery threshold; the upper root $x_+$ is stable.
  Starting below $x_-$ leads towards zero; starting above it leads towards $x_+$.
- **At $e_c$:** the roots meet. The steady state attracts from above but not from below
  (called *semistable*).
- **Above $e_c$:** no positive steady population remains.

Why? Factoring gives $f_B=-x(x-x_-)(x-x_+)/a$, whose signs are negative, positive,
negative across the three positive-population intervals. Also $f'_B(0)=-(1+e)<0$,
so zero stays stable even when harvesting stops.

**Example:** with $a=0.2$ and $e=0.4$, the recovery threshold is about $0.317K$
and the stable population is about $0.883K$. Here $e_c=0.8$.
The code below runs the shared experiment and shows these branches.

In [ ]:
import csv
import json
from pathlib import Path

from IPython.display import Image, Markdown, display

from mm_allee.wp02_experiment import run_experiment

ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "configs/wp02.json").exists() and (path / "src/mm_allee").exists()
)
# Run this experiment once.
config = json.loads((ROOT / "configs/wp02.json").read_text())
output = ROOT / "results/wp02-notebook"
manifest = run_experiment(config, output, ROOT)
display(Image(filename=str(output / "wp02_b_branches.png")))

### 2.2 Compare harvesting fairly

A fixed catch $H$ (tonnes/year) and an effort $h$ (per year) have different units.
We therefore use **two comparisons**:

1. **Same starting catch:** choose $H=hN_0$. Both rules remove the same amount initially;
   Model B's catch then changes with population.
2. **Same fraction of the limit:** use half of each model's own critical harvesting level.
   This compares equal distance from the collapse limit, but the initial catches differ.

For this example, $r=0.5$/year, $K=1000$ tonnes, $A=200$ tonnes, and the starting
population is 900 tonnes. We follow it for 60 years. The logistic model from Task 1
is included as a baseline.

**Mean catch = total catch / 60 years.** Total catch is the area under the catch-rate
curve: $\int_0^{60}hN(t)\,dt$ for B. For a fixed catch, harvesting stops at extinction.
The mean includes the years after shutdown.

The next table shows the important comparison results; the full CSV remains available.

In [ ]:
with (output / "wp02_comparison.csv").open() as f:
    rows = list(csv.DictReader(f))
labels = {"initial_catch": "Same starting catch", "threshold_fraction": "Half of each limit"}
outcomes = {"persistence": "Persists", "finite_extinction": "Collapses"}
lines = [
    "| Comparison | Model | Starting catch (t/year) | Mean catch (t/year) | Outcome |",
    "| --- | --- | --- | --- | --- |",
]
for row in rows:
    if float(row["initial_population"]) == 900 and float(row["fraction"]) == 0.5:
        lines.append(
            f"| {labels[row['protocol']]} | {row['model']} | "
            f"{float(row['initial_catch']):.2f} | {float(row['mean_catch']):.2f} | "
            f"{outcomes.get(row['theoretical_outcome'], row['theoretical_outcome'])} |"
        )
display(Markdown("\n".join(lines)))

**What this shows:** at the same starting catch (131.28 tonnes/year), A and B persist,
but the logistic population collapses because its maximum fixed catch is only
125 tonnes/year. At half of each model's limit, all three persist.

This does **not** prove that Allee growth is safer: the models have different growth
curves, and the conclusion depends on how the comparison is set up.

### 2.3 Does the population recover when harvesting falls?

**Method:** start at $N=K$, increase harvesting in steps to 1.3 times its critical
level, then decrease it to zero. Carry the population forward at every step—never
restart it at the turning point. Repeat with two dwell times (1 and 3 scaled time
units per ordinary step).

After harvesting returns to zero, compare:
- **No intervention:** let the population evolve on its own.
- **Restocking:** add $0.6K=600$ tonnes, shown separately by the dashed lines.

For A, reaching zero triggers harvest shutdown. For B, a positive population
approaches zero without reaching it in finite time; $x\leq10^{-6}$ is only a
smallness marker. The horizontal axis below is $\tau=rt$; here one unit equals two years.

In [ ]:
display(Image(filename=str(output / "wp02_history.png")))

### 2.4 Main conclusion

**Reducing harvesting does not guarantee recovery.** In both tested schedules,
the unassisted populations remain near zero. Once below the Allee threshold,
natural growth is negative even without harvesting. Adding 600 tonnes crosses
the threshold and allows recovery towards $K$.

This is **history dependence with a recovery barrier**, not a closed loop that
recovers by itself. The slower schedule also means longer exposure to harvesting,
so the experiment does not isolate the effect of speed alone.

**Checks and limits:** the experiment repeats the histories with tighter solver
settings; population differences are below $2\times10^{-9}$ in the recorded run.
Very small B populations have no claimed relative accuracy. Outcomes near an
uncertain starting-population boundary are labelled unresolved. The code, full
comparison table and run settings are saved by the experiment; no real-stock
prediction is claimed.

**Remember:** a smaller catch at low population helps reduce removal, but it does
not remove the Allee recovery threshold.


## 3. Critical Harvesting Rates and Extinction Analysis

In Sections 1 and 2, we studied the equilibrium points and stability of Models A and B. We now calculate the maximum harvesting rates that allow a positive equilibrium, determine which initial populations survive, and find the time taken for extinction when the population collapses.

We use the same population models and assumptions as before. All numerical results are obtained using synthetic data, with constant parameters and no random environmental effects.

Using the dimensionless variables introduced in Section 1,

$$
x=\frac{N}{K},\quad \tau=rt,\quad
a=\frac{A}{K},\quad q=\frac{H}{rK},\quad e=\frac{h}{r},
$$

the two models are

$$
\begin{aligned}
\text{Model A:}\quad &x'=G(x)-q,\\
\text{Model B:}\quad &x'=G(x)-ex,
\end{aligned}
$$

where

$$
G(x)=x(1-x)\left(\frac{x}{a}-1\right)
$$

and $x'=dx/d\tau$.

### 3.1 Critical Harvesting Rates

**Model A: Constant Harvesting**

At equilibrium, the natural population growth must balance the amount harvested. Hence, the maximum constant harvesting rate is obtained by finding the maximum value of $G(x)$.

Differentiating,

$$
G'(x)=\frac{-3x^2+2(1+a)x-a}{a}.
$$

Setting $G'(x)=0$, the larger stationary point is

$$
x_*=\frac{1+a+\sqrt{1-a+a^2}}{3}.
$$

This point lies between $a$ and $1$, and $G''(x_*)<0$, so it gives the maximum positive natural growth rate.

Therefore, the critical harvesting rate is

$$
q_c=G(x_*), \qquad H_c=rKq_c,
$$

where $H_c$ is the critical harvesting rate in the original units.

**Model B: Proportional Harvesting**

For Model B, the amount harvested depends on the population size. At a positive equilibrium,

$$
G(x)=ex.
$$

Dividing by $x$, we obtain

$$
e=\frac{G(x)}{x}=(1-x)\left(\frac{x}{a}-1\right)=P(x).
$$

Thus, unlike Model A, we find the maximum natural growth rate per unit population.

Differentiating,

$$
P'(x)=\frac{1+a-2x}{a}.
$$

Setting $P'(x)=0$ gives $x=(1+a)/2$. Since $P''(x)=-2/a<0$, this is a maximum. Substituting it into $P(x)$ gives

$$
e_c=\frac{(1-a)^2}{4a}.
$$

Converting back to the original parameters,

$$
h_c=\frac{r(K-A)^2}{4AK}.
$$

For harvesting rates above their respective critical values, neither model has a positive equilibrium. Below these values, the population may survive, depending on its initial size.

For comparison, the standard logistic model with constant harvesting has the critical rate $H_{c,\mathrm{logistic}}=rK/4$, as derived in Section 1.

Note that $H$ and $h$ have different units. We therefore cannot compare their numerical values directly.

The following code calculates the critical harvesting rates for the parameters used in our numerical examples.


In [ ]:
import csv
import json
import subprocess
import sys
from pathlib import Path

from IPython.display import Image, display

from mm_allee.models import Parameters
from mm_allee.thresholds import dimensional_thresholds

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "configs/wp03.json").exists() and (p / "src/mm_allee").exists()
)
config = json.loads((ROOT / "configs/wp03.json").read_text())
parameters = Parameters(**config["parameters"])
print("Synthetic parameters:", vars(parameters))
print("Dimensional thresholds:", dimensional_thresholds(parameters))


### 3.2 Effect of Initial Population on Survival

Even when harvesting is below the critical rate, the population may still become extinct if its initial size is too small.

To study this, we use the idea of a **basin of attraction**. A basin of attraction is the set of initial populations that approach the same equilibrium as time increases.

For Model A, when $0<q<q_c$, there are two positive equilibrium points, $x_-$ and $x_+$, where $x_-<x_+$.

By checking the sign of $x'=G(x)-q$, we find that:

- For $0<x<x_-$, the population decreases towards extinction.
- For $x_-<x<x_+$, the population increases towards $x_+$.
- For $x>x_+$, the population decreases towards $x_+$.

Thus, $x_-$ is an unstable equilibrium that separates extinction from survival, while $x_+$ is a stable equilibrium.

For Model B, the two positive equilibria are

$$
x_\pm=\frac{1+a\pm\sqrt{(1-a)^2-4ae}}{2}.
$$

The lower equilibrium is unstable and the upper equilibrium is stable. In addition, $x=0$ is always a stable equilibrium for Model B, since

$$
f'_B(0)=-(1+e)<0.
$$

The behaviour of both models at different harvesting levels is summarised below.

| Harvesting level | Model A | Model B |
|---|---|---|
| Zero harvesting | Below $a$, population approaches zero; above $a$, it approaches $1$ | Same behaviour as A |
| Below critical rate | Below $x_-$, population reaches zero; above $x_-$, it approaches $x_+$ | Below $x_-$, population approaches zero; above $x_-$, it approaches $x_+$ |
| At critical rate | Below the merged equilibrium, population reaches zero; above it, population approaches the merged equilibrium | Below the merged equilibrium, population approaches zero; above it, population approaches the merged equilibrium |
| Above critical rate | Every positive initial population eventually reaches zero | Every positive initial population approaches zero |

At the critical harvesting rate, the two positive equilibria merge into one. This equilibrium is **semistable**, meaning that the population approaches it from above but moves away from it when starting below.

An initial population placed exactly at an equilibrium remains there.

There is also an important difference between the models at zero population. For Model A with positive constant harvesting, $x=0$ is not an equilibrium because harvesting continues at a fixed rate. We therefore stop the numerical solution when the population first reaches zero.

For Model B, the amount harvested decreases with population, so zero remains an equilibrium.

The numerical calculations also account for rounding errors near unstable and semistable equilibria. Initial values extremely close to these boundaries are marked as unresolved rather than being classified incorrectly.



### 3.3 Calculating the Time to Extinction

For Model A, we can calculate the time taken for the population to reach zero using separation of variables.

We have

$$
\frac{dx}{d\tau}=G(x)-q.
$$

Separating variables,

$$
d\tau=\frac{dx}{G(x)-q}.
$$

Integrating from the initial population $x_0$ to zero and reversing the limits,

$$
\tau_{\mathrm{ext}}
=\int_{x_0}^{0}\frac{dx}{G(x)-q}
=\int_0^{x_0}\frac{dx}{q-G(x)}.
$$

This gives the extinction time in dimensionless units.

To convert it back to the original variables, recall that

$$
x=\frac{N}{K},\qquad \tau=rt,\qquad
q=\frac{H}{rK},\qquad G(x)=\frac{g(N)}{rK}.
$$

Therefore, $dx=dN/K$ and

$$
q-G(x)=\frac{H-g(N)}{rK}.
$$

Substituting these expressions into the integral,

$$
\tau_{\mathrm{ext}}
=\int_0^{N_0}
\frac{dN/K}{(H-g(N))/(rK)}
=r\int_0^{N_0}\frac{dN}{H-g(N)}.
$$

Since $\tau=rt$, the extinction time in the original time units is

$$
\boxed{
t_{\mathrm{ext}}
=\frac{\tau_{\mathrm{ext}}}{r}
=\int_0^{N_0}\frac{dN}{H-g(N)}
}
$$

Both expressions calculate the same extinction time. The first uses dimensionless variables, while the second uses the original population and time units.

This formula is valid only when the population actually reaches zero. We must first check the initial population and harvesting rate using the stability analysis from Section 3.2.

For a declining trajectory that reaches zero, the denominator remains positive throughout the integration interval. Hence, the integral gives a finite extinction time.

When $q>q_c$, we know that $G(x)\leq q_c$. Therefore,

$$
q-G(x)\geq q-q_c>0,
$$

which gives

$$
\tau_{\mathrm{ext}}\leq\frac{x_0}{q-q_c}.
$$

This confirms that extinction occurs in finite time above the critical constant harvesting rate.

For Model B, harvesting decreases as the population becomes smaller. A positive population may approach zero, but it does not reach exactly zero in finite time. The same applies to Model A when harvesting is zero.

Thus, **reaching zero in finite time and approaching zero as time increases are two different behaviours**.

### 3.4 Numerical Solution and Accuracy Checks

We use numerical methods to calculate extinction times and verify the analytical results.

The procedure is as follows:

1. Determine whether the initial population leads to extinction using the equilibrium and stability analysis.
2. For cases where extinction occurs, calculate the extinction time by numerical integration of the formula derived above.
3. Solve the differential equation using the adaptive Runge-Kutta method RK45. For Model A, stop the solution when the population first reaches zero.
4. Compare the results with another numerical solver, DOP853, to check their accuracy.

RK45 automatically adjusts its step size using two Runge-Kutta estimates of different orders. Their difference gives an estimate of the local numerical error, which is used to choose the next step size.

For our reference calculations, we use relative tolerance $10^{-9}$, absolute tolerance $10^{-11}$, and a maximum dimensionless step size of $0.1$.

To check the accuracy of the extinction times, we compare six representative cases using numerical integration, RK45 at two relative tolerances ($10^{-8}$ and $10^{-10}$), and DOP853 at $10^{-12}$.

We accept the results when the difference in dimensionless extinction times satisfies

$$
|\Delta\tau|\leq
2\times10^{-7}+5\times10^{-7}\tau_{\mathrm{ext}}.
$$

This verifies agreement for the tested cases, but it is not an exact error bound for every possible initial condition.

The next code cell runs the calculations, checks the numerical results, and generates the basin and extinction-time plots. It also generates the reference data used in the later PINN sections.

The optional sensitivity and maximum-yield calculations are not included in this notebook.


In [ ]:
outputs = {
    "analysis": ROOT / "results/wp03-notebook",
    "reference": ROOT / "results/wp03-reference-notebook",
}
commands = [
    ("analysis", "experiments/wp03/run.py", "configs/wp03.json", ["--core-only"]),
    ("reference", "experiments/wp03/generate_reference.py", "configs/wp03_reference.json", []),
]
for label, script, configuration, extra in commands:
    completed = subprocess.run(
        [
            sys.executable,
            str(ROOT / script),
            "--config",
            str(ROOT / configuration),
            "--output",
            str(outputs[label]),
            *extra,
        ],
        cwd=ROOT,
        check=True,
        capture_output=True,
        text=True,
    )
    print(completed.stdout.strip())


def read_rows(filename):
    with (outputs["analysis"] / filename).open(newline="") as stream:
        return list(csv.DictReader(stream))


evidence = read_rows("wp03_event_validation.csv")
assert all(row["status"] == "passed" for row in evidence)
print(
    "Largest recorded event difference in dimensionless time:",
    max(float(row["max_abs_error_tau"]) for row in evidence),
)
display(Image(filename=str(outputs["analysis"] / "wp03_basins.png")))
display(Image(filename=str(outputs["analysis"] / "wp03_extinction.png")))


**Interpretation of Results**

For the numerical examples, we use $r=0.7$, $K=100$, and $A=20$. The calculated critical values are

$$
N_*=70.55050,\qquad H_c=36.75966,\qquad h_c=0.56.
$$

For comparison, the critical constant harvesting rate for the standard logistic model is $17.5$.

**1. Model A: Basins of attraction (top-left)**

The horizontal axis shows the constant harvesting rate relative to its critical value, $H/H_c$, while the vertical axis shows the initial population relative to carrying capacity, $N_0/K$.

The blue region represents initial populations that approach a positive equilibrium. The orange region represents populations that reach zero in finite time.

The solid black curve is the unstable equilibrium, which separates survival from extinction. The dashed curve is the stable equilibrium. As harvesting increases, these two equilibria move closer together and merge at $H/H_c=1$.

Beyond this critical harvesting rate, the entire region is orange, indicating that every positive initial population eventually reaches zero.

**2. Model B: Basins of attraction (top-right)**

This graph shows the corresponding results for proportional harvesting, with $h/h_c$ on the horizontal axis.

The blue region again represents populations approaching a positive equilibrium. However, the yellow region represents populations that approach zero asymptotically, rather than reaching zero in finite time.

As in Model A, the unstable and stable positive equilibria merge at the critical harvesting rate. Beyond $h/h_c=1$, all positive initial populations decline towards zero.

Thus, both models have a critical harvesting limit, but their extinction behaviour differs because Model B's catch decreases with population size.

**3. Model A: Extinction time versus initial population (bottom-left)**

This graph shows how extinction time changes with the initial population for four different constant harvesting rates.

For harvesting below the critical rate, extinction time increases as the initial population approaches the unstable equilibrium from below. Above this boundary, the population survives, so no finite extinction time is plotted.

At the critical rate, the extinction time becomes very large as the initial population approaches the merged equilibrium from below.

When harvesting exceeds the critical rate, every positive initial population eventually reaches zero. However, extinction may still take a long time, particularly when harvesting is only slightly above the critical value.

**4. Model A: Extinction time versus constant catch (bottom-right)**

Here, the initial population is fixed at three values, $N_0=10$, $50$, and $90$, while the constant harvesting rate is varied.

For a fixed initial population that reaches zero, increasing the catch reduces the extinction time. However, the minimum catch needed to cause extinction depends on the initial population.

For $N_0=10$, the population is already below the Allee threshold, so it can decline even at relatively low harvesting rates.

For $N_0=50$, finite-time extinction occurs only once harvesting is sufficiently large to place the initial population below the unstable equilibrium.

For $N_0=90$, which is above the population level at the critical fold, extinction occurs only when $H>H_c$. The extinction time is particularly large just above the critical rate because the population declines slowly near the former equilibrium.

The vertical axes of both extinction-time graphs use a logarithmic scale.

**Numerical Verification**

The analysis produced 427 extinction records and 34 basin-classification cases. We independently compared extinction-time calculations using numerical quadrature, RK45, and DOP853.

All six validation cases passed, with the largest recorded difference in dimensionless extinction time approximately $4.56\times10^{-8}$.

Overall, the graphs show that survival depends on both the harvesting rate and the initial population. Even below the critical harvesting rate, a population may collapse if it starts below the unstable equilibrium. A long extinction time also does not necessarily imply stability.



### 3.5 Slow Decline Near the Critical Harvesting Rate

When the constant harvesting rate is slightly above its critical value, no positive equilibrium exists. However, the natural population growth can still be very close to the amount harvested near the population level $x_*$.

Since the difference between growth and harvesting is small, the population decreases very slowly in this region. It may remain close to the former equilibrium for a long time before eventually declining towards zero.

This behaviour is called a **bottleneck**.

The following graph shows population trajectories for three harvesting rates above the critical value. We observe that the closer the harvesting rate is to the critical value, the longer the population takes to collapse.

Thus, a population that changes very little over a short period is not necessarily stable. It may still become extinct if harvesting remains above the critical rate.

Here, we study this behaviour numerically rather than deriving an additional formula for the delay.


In [ ]:
display(Image(filename=str(outputs["analysis"] / "wp03_bottleneck.png")))


### 3.6 Synthetic Reference Data and Limitations

Along with the analytical calculations, we generated numerical solutions using RK45 for both harvesting models.

The reference dataset contains six synthetic trajectories with 606 sampled population values. These cover regular population behaviour, population decline or extinction, and cases close to the critical harvesting rates.

The parameter values, initial conditions, and numerical solver settings are recorded so that the results can be reproduced.

These numerical solutions are used as reference data in the later sections of the project. Section 4 studies the forward problem using Physics-Informed Neural Networks (PINNs), while Section 5 studies the inverse problem, where model parameters are estimated from simulated population observations.

The reference solutions also allow us to compare the PINN predictions with a conventional numerical method. The clean reference values used for evaluation are kept separate from the observations used for parameter fitting.

**Main conclusion.**

Our results show that both the harvesting rate and the initial population determine whether a population survives. Under constant harvesting, extinction can occur in finite time. Under proportional harvesting, a declining positive population approaches zero without reaching it in finite time.

We also find that extinction can be considerably delayed when the constant harvesting rate is just above its critical value. The numerical comparisons provide additional checks on the extinction-time calculations.

**Limitations.**

The models assume constant parameters and a well-mixed population. They do not account for environmental randomness, differences in age, or delays in population growth.

All experiments use synthetic data. Therefore, the results describe the behaviour predicted by these mathematical models and are not direct predictions for any particular real-world population.


In [ ]:
display(Image(filename=str(outputs["reference"] / "wp03_rk45_reference.png")))